In [2]:
pip install transformers datasets torch pandas scikit-learn accelerate peft huggingface_hub bitsandbytes

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [27]:
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    set_seed, 
    Trainer, 
    TrainingArguments, 
    pipeline, 
    EarlyStoppingCallback
)
from pathlib import Path
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report
import numpy as np
import pandas as pd
import torch
import json


# Dataset Preperation

In [3]:
main_dir = Path.cwd()

In [5]:
PATH_GAMETOX_TRAIN = "Existing_Datasets/GameTox/train.csv"
PATH_GAMETOX_VAL = "Existing_Datasets/GameTox/val.csv"

PATH_GAMETOX_TEST_TEXT = "Existing_Datasets/GameTox/test_index_text.csv"
PATH_GAMETOX_TEST_LABELS = "Existing_Datasets/GameTox/test_index_label.csv"

In [6]:
df_train = pd.read_csv( main_dir / PATH_GAMETOX_TRAIN )
df_val = pd.read_csv( main_dir / PATH_GAMETOX_VAL )

In [7]:
df_train["label"] = df_train["label"].astype(int)
df_val["label"] = df_val["label"].astype(int)

df_train = df_train[["message", "label"]]
df_val = df_val[["message", "label"]]

In [8]:
df_train = df_train[df_train['label'] < 4]
df_val = df_val[df_val['label'] < 4]

In [9]:
set_seed(42)

dataset_train = Dataset.from_pandas(df_train)
dataset_val = Dataset.from_pandas(df_val)

In [10]:
df_test_text = pd.read_csv(main_dir / PATH_GAMETOX_TEST_TEXT)
df_test_labels = pd.read_csv(main_dir / PATH_GAMETOX_TEST_LABELS)

In [11]:
df_test = pd.concat([df_test_text['message'], df_test_labels['label']], axis=1)
df_test = df_test[df_test['label'] < 4]
df_test['label'] = df_test['label'].astype(int)

# Evaluation Function

In [12]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    n_classes = logits.shape[1]

    report = classification_report(
        labels, 
        predictions, 
        output_dict=True, 
        zero_division=0,
        digits=4
    )
    
    per_label_precision = []
    per_label_recall = []
    per_label_f1 = []
    
    for i in range(n_classes):
        per_label_precision.append(report[str(i)]['precision'])
        per_label_recall.append(report[str(i)]['recall'])
        per_label_f1.append(report[str(i)]['f1-score'])

    return {
        "macro_F1":
            f1_score(
                labels,
                predictions,
                average="macro"
            ),

        "per_label_precision": per_label_precision,
        "per_label_recall": per_label_recall,
        "per_label_f1": per_label_f1
    }

# Training Arguments

In [13]:
def training_args(local_save_dir):
    return TrainingArguments(
        output_dir=local_save_dir,

        learning_rate=2e-5,

        per_device_train_batch_size=32,

        weight_decay=0.01,

        metric_for_best_model="macro_F1",
        greater_is_better=True, 

        eval_strategy="steps",

        warmup_steps=0.1,

        save_strategy="steps",

        num_train_epochs=3,

        save_total_limit=2
)

# HateBERT

In [12]:
MODEL_NAME_HATEBERT = "GroNLP/hateBERT"

In [13]:
tokenizer_hatebert = AutoTokenizer.from_pretrained(MODEL_NAME_HATEBERT)

model_hatebert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_HATEBERT,
    num_labels=4,
    ignore_mismatched_sizes=True
)

config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `2`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
def tokenize_hatebert(batch):
    return tokenizer_hatebert(
        batch["message"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [16]:
tokenized_dataset_train = dataset_train.map(
    tokenize_hatebert,
    batched=True
)

Map:   0%|          | 0/42875 [00:00<?, ? examples/s]

In [17]:
tokenized_dataset_val = dataset_val.map(
    tokenize_hatebert,
    batched=True
)

Map:   0%|          | 0/5357 [00:00<?, ? examples/s]

In [30]:
trainer_hatebert = Trainer(
    model=model_hatebert,

    args=training_args('./HateBERT-GameTox'),

    train_dataset=tokenized_dataset_train,

    eval_dataset=tokenized_dataset_val,

    compute_metrics=compute_metrics,

    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [31]:
trainer_hatebert.train()

[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Step,Training Loss,Validation Loss,Macro F1,Per Label Precision,Per Label Recall,Per Label F1
500,0.170922,0.434908,0.597964,"[0.9556142034548945, 0.6023506366307542, 0.5128205128205128, 0.6666666666666666]","[0.9158427224649345, 0.831081081081081, 0.3418803418803419, 0.23529411764705882]","[0.9353058588704943, 0.6984667802385008, 0.41025641025641024, 0.34782608695652173]"
1000,0.169962,0.400864,0.621850,"[0.930186170212766, 0.7671033478893741, 0.5845070422535211, 0.5625]","[0.965049436652104, 0.7121621621621622, 0.3547008547008547, 0.2647058823529412]","[0.9472971447917842, 0.7386124737210932, 0.44148936170212766, 0.36]"
1500,0.148376,0.446597,0.624682,"[0.950407450523865, 0.7155727155727156, 0.4312267657992565, 0.5625]","[0.9386065762244195, 0.7513513513513513, 0.49572649572649574, 0.2647058823529412]","[0.94447015270708, 0.7330257086354647, 0.46123260437375746, 0.36]"
2000,0.201296,0.369968,0.633043,"[0.9486942454356367, 0.7083839611178615, 0.4943820224719101, 0.4482758620689655]","[0.9438951483099564, 0.7878378378378378, 0.37606837606837606, 0.38235294117647056]","[0.946288612263716, 0.746001279590531, 0.42718446601941745, 0.4126984126984127]"
2500,0.207077,0.352513,0.640975,"[0.9449646037908198, 0.7367021276595744, 0.494949494949495, 0.4642857142857143]","[0.9514830995631179, 0.7486486486486487, 0.4188034188034188, 0.38235294117647056]","[0.9482126489459212, 0.7426273458445041, 0.4537037037037037, 0.41935483870967744]"
3000,0.176352,0.389541,0.634037,"[0.9516242112643141, 0.7022332506203474, 0.4462809917355372, 0.43333333333333335]","[0.9363071970567947, 0.7648648648648648, 0.46153846153846156, 0.38235294117647056]","[0.9439035697728326, 0.7322121604139715, 0.453781512605042, 0.40625]"
3500,0.158596,0.385719,0.640959,"[0.9455922865013774, 0.7157232704402515, 0.4942528735632184, 0.46875]","[0.947114279144631, 0.768918918918919, 0.36752136752136755, 0.4411764705882353]","[0.9463526708788053, 0.7413680781758958, 0.4215686274509804, 0.45454545454545453]"
4000,0.152552,0.392860,0.635412,"[0.9459397285484242, 0.7217948717948718, 0.47029702970297027, 0.4642857142857143]","[0.9455047137272936, 0.7608108108108108, 0.405982905982906, 0.38235294117647056]","[0.9457221711131555, 0.7407894736842106, 0.43577981651376146, 0.41935483870967744]"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4000, training_loss=0.17314175224304199, metrics={'train_runtime': 2853.9019, 'train_samples_per_second': 45.07, 'train_steps_per_second': 1.409, 'total_flos': 3.367618869448704e+16, 'train_loss': 0.17314175224304199, 'epoch': 2.9850746268656714})

In [32]:
print(trainer_hatebert.state.best_model_checkpoint)

./HateBERT-GameTox/checkpoint-2500


In [14]:
model_hatebert_from_checkpoint = AutoModelForSequenceClassification.from_pretrained(
    main_dir / "HateBERT-GameTox" / "checkpoint-2500"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [15]:
classifier = pipeline("text-classification", model=model_hatebert_from_checkpoint, tokenizer=tokenizer_hatebert)

In [16]:
preds = classifier(df_test['message'].tolist())
preds = [(model_hatebert_from_checkpoint.config.label2id[pred['label']]) for pred in preds]

In [17]:
print(classification_report(df_test['label'], preds, digits=4, zero_division=True))

              precision    recall  f1-score   support

           0     0.9462    0.9612    0.9536      4351
           1     0.7698    0.7615    0.7656       742
           2     0.5027    0.4000    0.4455       235
           3     0.6087    0.3889    0.4746        36

    accuracy                         0.9051      5364
   macro avg     0.7068    0.6279    0.6598      5364
weighted avg     0.9001    0.9051    0.9021      5364



# Detoxify

In [37]:
MODEL_NAME_DETOXIFY = "unitary/toxic-bert"

In [41]:
tokenizer_detoxify = AutoTokenizer.from_pretrained(MODEL_NAME_DETOXIFY)

model_detoxify = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_DETOXIFY,
    num_labels=4,
    ignore_mismatched_sizes=True,
    problem_type="single_label_classification"
)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `6`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6, 768]) vs model:torch.Size([4, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [42]:
def tokenize_detoxify(batch):
    return tokenizer_detoxify(
        batch["message"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [47]:
tokenized_dataset_train = dataset_train.map(
    tokenize_detoxify,
    batched=True
)

Map:   0%|          | 0/42875 [00:00<?, ? examples/s]

In [48]:
tokenized_dataset_val = dataset_val.map(
    tokenize_detoxify,
    batched=True
)

Map:   0%|          | 0/5357 [00:00<?, ? examples/s]

In [49]:
trainer_detoxify = Trainer(
    model=model_detoxify,

    args=training_args('./Detoxify-GameTox'),

    train_dataset=tokenized_dataset_train,

    eval_dataset=tokenized_dataset_val,

    compute_metrics=compute_metrics,

    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [50]:
trainer_detoxify.train()

[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Step,Training Loss,Validation Loss,Macro F1,Per Label Precision,Per Label Recall,Per Label F1
500,0.499933,0.325544,0.518551,"[0.9381443298969072, 0.7085927770859277, 0.6847826086956522, 0.0]","[0.9625201195677168, 0.768918918918919, 0.2692307692307692, 0.0]","[0.9501759164680513, 0.7375243033052495, 0.38650306748466257, 0.0]"
1000,0.320953,0.311226,0.583812,"[0.9333185152256057, 0.7661764705882353, 0.5260115606936416, 0.8]","[0.9655093124856289, 0.7040540540540541, 0.3888888888888889, 0.11764705882352941]","[0.9491410488245932, 0.7338028169014085, 0.44717444717444715, 0.20512820512820512]"
1500,0.279866,0.318438,0.638333,"[0.9509668508287292, 0.7083333333333334, 0.5348837209302325, 0.48]","[0.9498735341457807, 0.7810810810810811, 0.39316239316239315, 0.35294117647058826]","[0.9504198780628091, 0.7429305912596401, 0.45320197044334976, 0.4067796610169492]"
2000,0.251518,0.320089,0.651781,"[0.9507514450867052, 0.707673568818514, 0.5469613259668509, 0.4666666666666667]","[0.9455047137272936, 0.7851351351351351, 0.4230769230769231, 0.4117647058823529]","[0.9481208208439014, 0.7443946188340808, 0.4771084337349398, 0.4375]"
2500,0.240263,0.301200,0.665092,"[0.9488233950194197, 0.7210460772104608, 0.5660377358490566, 0.7222222222222222]","[0.9549321683145551, 0.7824324324324324, 0.38461538461538464, 0.38235294117647056]","[0.9518679807471923, 0.7504860661049902, 0.4580152671755725, 0.5]"
3000,0.206584,0.328837,0.654821,"[0.9455456823877877, 0.7443105756358769, 0.5177664974619289, 0.5416666666666666]","[0.9542423545642676, 0.7513513513513513, 0.4358974358974359, 0.38235294117647056]","[0.9498741130693522, 0.7478143913920645, 0.4733178654292343, 0.4482758620689655]"
3500,0.196823,0.325220,0.655702,"[0.9462734339277549, 0.7176913425345044, 0.5337423312883436, 0.6086956521739131]","[0.9517130374798805, 0.772972972972973, 0.3717948717948718, 0.4117647058823529]","[0.9489854407887195, 0.7443070917371503, 0.43828715365239296, 0.49122807017543857]"
4000,0.192015,0.326893,0.661696,"[0.9458409506398537, 0.7252888318356868, 0.5222222222222223, 0.6363636363636364]","[0.9517130374798805, 0.7635135135135135, 0.4017094017094017, 0.4117647058823529]","[0.9487679083094556, 0.7439104674127716, 0.45410628019323673, 0.5]"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4000, training_loss=0.27349424743652345, metrics={'train_runtime': 2852.4613, 'train_samples_per_second': 45.093, 'train_steps_per_second': 1.409, 'total_flos': 3.367618869448704e+16, 'train_loss': 0.27349424743652345, 'epoch': 2.9850746268656714})

In [51]:
print(trainer_detoxify.state.best_model_checkpoint)

./Detoxify-GameTox/checkpoint-2500


In [52]:
model_detoxify_checkpoint = AutoModelForSequenceClassification.from_pretrained(
    main_dir / "Detoxify-GameTox" / "checkpoint-2500"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [53]:
classifier = pipeline("text-classification", model=model_detoxify_checkpoint, tokenizer=tokenizer_detoxify)

In [54]:
preds = classifier(df_test['message'].tolist())
preds = [(model_detoxify_checkpoint.config.label2id[pred['label']]) for pred in preds]

In [55]:
print(classification_report(df_test['label'], preds, digits=4, zero_division=True))

              precision    recall  f1-score   support

           0     0.9484    0.9639    0.9561      4351
           1     0.7591    0.7898    0.7741       742
           2     0.5972    0.3660    0.4538       235
           3     0.5769    0.4167    0.4839        36

    accuracy                         0.9100      5364
   macro avg     0.7204    0.6341    0.6670      5364
weighted avg     0.9044    0.9100    0.9058      5364



# BERT

In [5]:
MODEL_NAME_BERT = "google-bert/bert-base-uncased"

In [19]:
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME_BERT)

model_bert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_BERT,
    num_labels=4,
    ignore_mismatched_sizes=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize_bert(batch):
    return tokenizer_bert(
        batch["message"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [60]:
tokenized_dataset_train = dataset_train.map(
    tokenize_bert,
    batched=True
)

Map:   0%|          | 0/42875 [00:00<?, ? examples/s]

In [61]:
tokenized_dataset_val = dataset_val.map(
    tokenize_bert,
    batched=True
)

Map:   0%|          | 0/5357 [00:00<?, ? examples/s]

In [62]:
trainer_bert = Trainer(
    model=model_bert,

    args=training_args('./BERT-GameTox'),

    train_dataset=tokenized_dataset_train,

    eval_dataset=tokenized_dataset_val,

    compute_metrics=compute_metrics,

    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [63]:
trainer_bert.train()

[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Step,Training Loss,Validation Loss,Macro F1,Per Label Precision,Per Label Recall,Per Label F1
500,0.570065,0.344192,0.497916,"[0.9291565198140359, 0.7035110533159948, 0.704225352112676, 0.0]","[0.965049436652104, 0.731081081081081, 0.21367521367521367, 0.0]","[0.94676291450485, 0.7170311464546058, 0.32786885245901637, 0.0]"
1000,0.331176,0.331509,0.529492,"[0.9302067126028006, 0.7626112759643917, 0.5054347826086957, 0.0]","[0.9622901816509543, 0.6945945945945946, 0.3974358974358974, 0.0]","[0.9459764918625678, 0.727015558698727, 0.4449760765550239, 0.0]"
1500,0.289402,0.318512,0.540047,"[0.947138588830154, 0.695078031212485, 0.5606936416184971, 0.0]","[0.9475741549781559, 0.7824324324324324, 0.41452991452991456, 0.0]","[0.9473563218390805, 0.7361729179910999, 0.47665847665847666, 0.0]"
2000,0.264794,0.322141,0.641134,"[0.945014830025097, 0.7332474226804123, 0.56, 0.4782608695652174]","[0.9524028512301679, 0.768918918918919, 0.4188034188034188, 0.3235294117647059]","[0.9486944571690334, 0.7506596306068601, 0.4792176039119804, 0.38596491228070173]"
2500,0.249957,0.300540,0.645906,"[0.945872185581078, 0.7450722733245729, 0.532608695652174, 0.6666666666666666]","[0.9563117958151299, 0.7662162162162162, 0.4188034188034188, 0.29411764705882354]","[0.9510633432426252, 0.7554963357761493, 0.4688995215311005, 0.40816326530612246]"
3000,0.213162,0.329496,0.650887,"[0.9460817911811743, 0.7327249022164276, 0.5544041450777202, 0.55]","[0.9521729133134054, 0.7594594594594595, 0.45726495726495725, 0.3235294117647059]","[0.9491175796470318, 0.7458526874585268, 0.5011709601873536, 0.4074074074074074]"
3500,0.202193,0.330298,0.663469,"[0.9434476493300022, 0.7404479578392622, 0.5705882352941176, 0.56]","[0.9551621062313176, 0.7594594594594595, 0.41452991452991456, 0.4117647058823529]","[0.9492687385740403, 0.7498332221480988, 0.4801980198019802, 0.4745762711864407]"
4000,0.203747,0.324615,0.648213,"[0.9444570908263146, 0.7413793103448276, 0.5439560439560439, 0.4642857142857143]","[0.9540124166475051, 0.7554054054054054, 0.4230769230769231, 0.38235294117647056]","[0.9492107069320521, 0.7483266398929049, 0.47596153846153844, 0.41935483870967744]"
4020,0.203747,0.324550,0.649807,"[0.9444570908263146, 0.7403973509933774, 0.5439560439560439, 0.48148148148148145]","[0.9540124166475051, 0.7554054054054054, 0.4230769230769231, 0.38235294117647056]","[0.9492107069320521, 0.7478260869565218, 0.47596153846153844, 0.4262295081967213]"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4020, training_loss=0.29007197879440155, metrics={'train_runtime': 2904.4283, 'train_samples_per_second': 44.286, 'train_steps_per_second': 1.384, 'total_flos': 3.3843267214848e+16, 'train_loss': 0.29007197879440155, 'epoch': 3.0})

In [64]:
print(trainer_bert.state.best_model_checkpoint)

./BERT-GameTox/checkpoint-3500


In [20]:
model_bert_checkpoint = AutoModelForSequenceClassification.from_pretrained(
    main_dir / "BERT-GameTox" / "checkpoint-3500"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [21]:
classifier = pipeline("text-classification", model=model_bert_checkpoint, tokenizer=tokenizer_bert)

In [22]:
preds = classifier(df_test['message'].tolist())
preds = [(model_bert_checkpoint.config.label2id[pred['label']]) for pred in preds]

In [23]:
print(classification_report(df_test['label'], preds, digits=4, zero_division=True))

              precision    recall  f1-score   support

           0     0.9466    0.9612    0.9538      4351
           1     0.7649    0.7763    0.7706       742
           2     0.5329    0.3787    0.4428       235
           3     0.6154    0.4444    0.5161        36

    accuracy                         0.9066      5364
   macro avg     0.7150    0.6402    0.6708      5364
weighted avg     0.9011    0.9066    0.9031      5364



# RoBERTa

In [ ]:
MODEL_NAME_ROBERTA = "FacebookAI/roberta-base"

In [ ]:
tokenizer_roberta = AutoTokenizer.from_pretrained(MODEL_NAME_ROBERTA)

model_roberta = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_ROBERTA,
    num_labels=4,
    ignore_mismatched_sizes=True
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize_roberta(batch):
    return tokenizer_roberta(
        batch["message"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [ ]:
tokenized_dataset_train = dataset_train.map(
    tokenize_roberta,
    batched=True
)

Map:   0%|          | 0/42875 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset_val = dataset_val.map(
    tokenize_roberta,
    batched=True
)

Map:   0%|          | 0/5357 [00:00<?, ? examples/s]

In [ ]:
trainer_roberta = Trainer(
    model=model_roberta,

    args=training_args('./RoBERTa-GameTox'),

    train_dataset=tokenized_dataset_train,

    eval_dataset=tokenized_dataset_val,

    compute_metrics=compute_metrics,

    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
trainer_roberta.train()

[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Step,Training Loss,Validation Loss,Macro F1,Per Label Precision,Per Label Recall,Per Label F1
500,0.558099,0.351246,0.504084,"[0.9323899371069182, 0.6906832298136646, 0.59, 0.0]","[0.9544722924810302, 0.7513513513513513, 0.25213675213675213, 0.0]","[0.9433018975116464, 0.7197411003236246, 0.3532934131736527, 0.0]"
1000,0.346438,0.326902,0.536026,"[0.9379512635379061, 0.7422402159244265, 0.5163043478260869, 0.0]","[0.955851919981605, 0.7432432432432432, 0.405982905982906, 0.0]","[0.9468169912310671, 0.7427413909520594, 0.45454545454545453, 0.0]"
1500,0.308829,0.327646,0.536515,"[0.9422769792379648, 0.7144654088050314, 0.5307262569832403, 0.0]","[0.9496435962290182, 0.7675675675675676, 0.405982905982906, 0.0]","[0.9459459459459459, 0.7400651465798046, 0.4600484261501211, 0.0]"
2000,0.291938,0.344306,0.552560,"[0.9515108924806747, 0.6778900112233446, 0.5153061224489796, 1.0]","[0.93400781788917, 0.8162162162162162, 0.43162393162393164, 0.029411764705882353]","[0.9426781155720585, 0.7406499080318822, 0.4697674418604651, 0.05714285714285714]"
2500,0.269914,0.316861,0.564119,"[0.9372338040798027, 0.7575757575757576, 0.5389221556886228, 0.6666666666666666]","[0.9613704299839043, 0.7432432432432432, 0.38461538461538464, 0.058823529411764705]","[0.9491486946651533, 0.7503410641200545, 0.4488778054862843, 0.10810810810810811]"
3000,0.247197,0.328573,0.589957,"[0.950277520814061, 0.7075, 0.4845814977973568, 0.6666666666666666]","[0.9448148999770062, 0.7648648648648648, 0.4700854700854701, 0.11764705882352941]","[0.9475383373688459, 0.7350649350649351, 0.4772234273318872, 0.2]"
3500,0.238645,0.324827,0.605430,"[0.9480937069361507, 0.7120291616038882, 0.5443786982248521, 0.5454545454545454]","[0.9491837203954933, 0.7918918918918919, 0.39316239316239315, 0.17647058823529413]","[0.9486384005515339, 0.7498400511836213, 0.456575682382134, 0.26666666666666666]"
4000,0.234031,0.323316,0.615218,"[0.9486238532110092, 0.732824427480916, 0.5148514851485149, 0.6666666666666666]","[0.951023223729593, 0.7783783783783784, 0.4444444444444444, 0.17647058823529413]","[0.9498220231943966, 0.7549148099606815, 0.47706422018348627, 0.27906976744186046]"
4020,0.234031,0.323221,0.615218,"[0.9486238532110092, 0.732824427480916, 0.5148514851485149, 0.6666666666666666]","[0.951023223729593, 0.7783783783783784, 0.4444444444444444, 0.17647058823529413]","[0.9498220231943966, 0.7549148099606815, 0.47706422018348627, 0.27906976744186046]"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4020, training_loss=0.31146160464974776, metrics={'train_runtime': 2876.903, 'train_samples_per_second': 44.71, 'train_steps_per_second': 1.397, 'total_flos': 3.3843267214848e+16, 'train_loss': 0.31146160464974776, 'epoch': 3.0})

In [ ]:
print(trainer_roberta.state.best_model_checkpoint)

./DeBERTa-GameTox/checkpoint-4000


In [ ]:
model_roberta_checkpoint = AutoModelForSequenceClassification.from_pretrained(
    main_dir / "RoBERTa-GameTox" / "checkpoint-4000"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
classifier = pipeline("text-classification", model=model_roberta_checkpoint, tokenizer=tokenizer_roberta)

In [38]:
preds = classifier(df_test['message'].tolist())
preds = [(model_roberta_checkpoint.config.label2id[pred['label']]) for pred in preds]

In [39]:
print(classification_report(df_test['label'], preds, digits=4, zero_division=True))

              precision    recall  f1-score   support

           0     0.9470    0.9605    0.9537      4351
           1     0.7673    0.7911    0.7790       742
           2     0.5314    0.3957    0.4537       235
           3     0.7273    0.2222    0.3404        36

    accuracy                         0.9073      5364
   macro avg     0.7432    0.5924    0.6317      5364
weighted avg     0.9024    0.9073    0.9035      5364



# Baseline Classifer (TF-IDF Logistic Regression)

In [69]:
from sklearn.feature_extraction.text import TfidfVectorizer

def create_tfidf_vectorizer():
    return TfidfVectorizer(
        ngram_range=(1, 3),
        max_features=100000,
        sublinear_tf=True,
        analyzer='word',
        lowercase=False,
        min_df=2,
        max_df=0.8,
        norm='l2',
        use_idf=True
    )

In [70]:
from sklearn.linear_model import LogisticRegression

def train_baseline_model(X_train, y_train, X_val=None, y_val=None):
    
    model = LogisticRegression(
        max_iter=2000,
        random_state=42,
        C=0.5,
        class_weight='balanced',
        solver='lbfgs',
        n_jobs=-1,
        verbose=1
    )
    
    model.fit(X_train, y_train)
    
    if X_val is not None:
        val_pred = model.predict(X_val)
        val_f1 = f1_score(y_val, val_pred, average='macro')
        print(f"\nValidation Performance:")
        print(f"  Macro F1: {val_f1:.4f}")
    
    return model

In [72]:
vectorizer = create_tfidf_vectorizer()

X_train_tfidf = vectorizer.fit_transform(df_train['message'])
X_val_tfidf = vectorizer.transform(df_val['message'])

In [73]:
model_basline = train_baseline_model(X_train_tfidf, df_train['label'], X_val_tfidf, df_val['label'])

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.



Validation Performance:
  Macro F1: 0.5179


In [74]:
X_test_tfidf = vectorizer.transform(df_test['message'])

In [75]:
y_pred = model_basline.predict(X_test_tfidf)
    
print(classification_report(df_test['label'], y_pred, digits=4))

              precision    recall  f1-score   support

           0     0.9468    0.8582    0.9003      4351
           1     0.6548    0.6442    0.6495       742
           2     0.2442    0.5787    0.3434       235
           3     0.0977    0.3611    0.1538        36

    accuracy                         0.8130      5364
   macro avg     0.4859    0.6106    0.5118      5364
weighted avg     0.8699    0.8130    0.8362      5364



# Final Robustness Analysis on other games

In [6]:
model_best = AutoModelForSequenceClassification.from_pretrained(
    main_dir / "BERT-GameTox" / "checkpoint-3500"
)
tokenizer_best = AutoTokenizer.from_pretrained(MODEL_NAME_BERT)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
classifier_best = pipeline("text-classification", model=model_best, tokenizer=tokenizer_best)

In [10]:
df_test_game_one = pd.read_csv(main_dir / "text_data" / "Steam" / "Marvel_Rivals.csv")
df_test_game_one = df_test_game_one.head(500)

In [22]:
df_test_game_one.tail()

,text
495,"They never will, its by design and the people ..."
496,hey when we will get guest characters in marve...
497,CAN YOUY PLEASE REMOVE SG AND ANY CHARACTER TH...
498,Now if you could just make the game skill based
499,Still not removed bot games from quickplay.


In [13]:
preds = classifier_best(df_test_game_one['text'].tolist(), truncation=True, max_length=512)
preds = [(model_best.config.label2id[pred['label']]) for pred in preds]

In [28]:
with open('llm_output_game_one.json', 'r') as file:
    llm_output = json.load(file)

In [29]:
llm_labels = [o['label'] for o in llm_output]

In [30]:
print(classification_report(llm_labels, preds, digits=4))

              precision    recall  f1-score   support

           0     0.7109    0.8667    0.7811       315
           1     0.2093    0.1452    0.1714        62
           2     0.4118    0.2545    0.3146       110
           3     0.4000    0.1538    0.2222        13

    accuracy                         0.6240       500
   macro avg     0.4330    0.3551    0.3723       500
weighted avg     0.5748    0.6240    0.5884       500



In [31]:
df_test_game_two = pd.read_csv(main_dir / "text_data" / "Steam" / "Dead_by_Daylight.csv")
df_test_game_two = df_test_game_two.head(500)

In [32]:
df_test_game_two.tail()

,text
495,From what I observed A lot of players live in ...
496,As a new player with quite literally 4ish hour...
497,"thats kinda part of the learning curve, You ha..."
498,It's kind of always been a given that starting...
499,high iq post


In [33]:
preds = classifier_best(df_test_game_two['text'].tolist(), truncation=True, max_length=512)
preds = [(model_best.config.label2id[pred['label']]) for pred in preds]

In [34]:
with open('llm_output_game_two.json', 'r') as file:
    llm_output = json.load(file)

llm_labels = [o['label'] for o in llm_output]

In [35]:
print(classification_report(llm_labels, preds, digits=4))

              precision    recall  f1-score   support

           0     0.8412    0.8853    0.8627       401
           1     0.4211    0.1481    0.2192        54
           2     0.1552    0.2093    0.1782        43
           3     1.0000    0.5000    0.6667         2

    accuracy                         0.7460       500
   macro avg     0.6044    0.4357    0.4817       500
weighted avg     0.7375    0.7460    0.7335       500



In [36]:
df_test_game_three = pd.read_csv(main_dir / "text_data" / "Steam" / "Team_Fortress_2.csv")
df_test_game_three = df_test_game_three.head(500)

In [37]:
df_test_game_three.tail()

,text
495,Every time i come back to these forums i kinda...
496,I think they're just recycling same ♥♥♥♥ over ...
497,"So what you're saying is, in Team Fortress 2, ..."
498,The replies and awards say the opposite! Chang...
499,Thats why they put Scout in the game


In [38]:
preds = classifier_best(df_test_game_three['text'].tolist(), truncation=True, max_length=512)
preds = [(model_best.config.label2id[pred['label']]) for pred in preds]

In [39]:
with open('llm_output_game_three.json', 'r') as file:
    llm_output = json.load(file)

llm_labels = [o['label'] for o in llm_output]

In [40]:
print(classification_report(llm_labels, preds, digits=4))

              precision    recall  f1-score   support

           0     0.7422    0.8584    0.7961       332
           1     0.5397    0.3656    0.4359        93
           2     0.2059    0.1346    0.1628        52
           3     0.3158    0.2609    0.2857        23

    accuracy                         0.6640       500
   macro avg     0.4509    0.4049    0.4201       500
weighted avg     0.6291    0.6640    0.6398       500

